# Chapter 19 — Agent Architecture and Planning
## Building Agents That Actually *Think*

Most tutorials show agents that *react* — they see a problem and grab a tool.
Real production agents **plan ahead**, recover from failures, and learn from mistakes.

This notebook teaches four planning architectures used in real networking automation:

| Pattern | What It Does | Best For |
|---------|-------------|----------|
| **HTN + DAG** | Decompose big tasks into dependency graphs | Structured change management |
| **ReAct + Lookahead** | Form hypothesis first, then gather evidence | Network troubleshooting |
| **Plan-and-Execute** | Separate planning from execution, review gates | High-risk changes |
| **Reflexion** | Learn from failure in words, not weights | Self-improving diagnostic agents |

> **No GPU needed.** Every demo runs on the free Anthropic API (Haiku for fast ops, Sonnet for planning).

### Install
```
pip install anthropic
```


## Setup — API Client and Shared Utilities

In [ ]:
import os, json, re, time
from anthropic import Anthropic
from collections import defaultdict, deque

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()  # reads ANTHROPIC_API_KEY from env

HAIKU  = "claude-haiku-4-5-20251001"   # fast, cheap — for sub-tasks
SONNET = "claude-sonnet-4-20250514"           # smarter — for planning & reflection

def ask(prompt, model=HAIKU, system="You are a helpful network automation assistant."):
    """Single-turn helper — returns plain text."""
    r = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    return r.content[0].text.strip()

print("Client ready.")
print(f"Fast model : {HAIKU}")
print(f"Smart model: {SONNET}")


---
## Demo 1 — HTN Planning: Decompose Tasks into a DAG

**Hierarchical Task Network (HTN)** planning breaks a high-level goal into a
Directed Acyclic Graph (DAG) of sub-tasks with dependency edges.

Think of it like **OSPF route calculation** — you can't advertise a route until
you've computed the SPF tree. Some tasks *must* happen before others.

```
Goal: "Deploy new OSPF area 10"
         │
    ┌────┴─────┐
  backup     validate
    │           │
    └────┬──────┘
      configure
         │
       verify
```

The DAG ensures we never run `configure` before `backup` and `validate` are done.


In [ ]:
# ── HTN Planning: Task Decomposition + DAG Execution ──────────────────────────

class Task:
    def __init__(self, name, description, depends_on=None):
        self.name       = name
        self.description = description
        self.depends_on = depends_on or []   # list of task names this task needs first
        self.status     = "pending"          # pending → running → done / failed
        self.result     = None

class NetworkPlanningAgent:
    """
    Decomposes a high-level network goal into a DAG of tasks,
    then executes them in topological order.
    """

    def __init__(self):
        self.tasks = {}

    # ── Step 1: Ask the LLM to decompose the goal ────────────────────────────
    def decompose_task(self, goal: str) -> list[Task]:
        prompt = f"""You are a network change management system.

Decompose this goal into 5-7 atomic sub-tasks with dependencies:
Goal: {goal}

Return ONLY a JSON array. Each element has:
  - "name": short snake_case identifier
  - "description": one sentence
  - "depends_on": list of name strings that must complete first

Example format:
[
  {{"name": "backup_config", "description": "Back up current running config.", "depends_on": []}},
  {{"name": "validate_syntax", "description": "Validate new config syntax.", "depends_on": ["backup_config"]}}
]
"""
        raw = ask(prompt, model=SONNET)
        # Extract JSON array
        match = re.search(r'\[.*\]', raw, re.DOTALL)
        if not match:
            raise ValueError(f"No JSON found in: {raw[:200]}")
        data = json.loads(match.group())
        tasks = [Task(t["name"], t["description"], t.get("depends_on", [])) for t in data]
        for t in tasks:
            self.tasks[t.name] = t
        return tasks

    # ── Step 2: Topological sort (Kahn's algorithm) ──────────────────────────
    def topological_order(self) -> list[str]:
        in_degree = {name: 0 for name in self.tasks}
        adj       = defaultdict(list)
        for name, task in self.tasks.items():
            for dep in task.depends_on:
                adj[dep].append(name)
                in_degree[name] += 1

        queue  = deque([n for n, d in in_degree.items() if d == 0])
        result = []
        while queue:
            node = queue.popleft()
            result.append(node)
            for neighbor in adj[node]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)
        return result

    # ── Step 3: Simulate execution ───────────────────────────────────────────
    def execute(self, goal: str):
        print(f"Goal: {goal}")
        print("=" * 60)

        tasks = self.decompose_task(goal)
        print(f"Decomposed into {len(tasks)} tasks:\n")
        for t in tasks:
            deps = ", ".join(t.depends_on) if t.depends_on else "none"
            print(f"  [{t.name}]  depends_on: {deps}")
            print(f"    → {t.description}")

        order = self.topological_order()
        print(f"\nExecution order (topological): {' → '.join(order)}")
        print()

        for name in order:
            task = self.tasks[name]
            # Check all dependencies completed
            for dep in task.depends_on:
                if self.tasks[dep].status != "done":
                    task.status = "failed"
                    print(f"  ✗ {name}  BLOCKED — {dep} not done")
                    continue

            task.status = "running"
            print(f"  ▶ {name}  running…")
            time.sleep(0.1)   # simulate work
            task.status = "done"
            task.result = f"Completed: {task.description}"
            print(f"  ✓ {name}  done")

        done  = sum(1 for t in self.tasks.values() if t.status == "done")
        total = len(self.tasks)
        print(f"\nResult: {done}/{total} tasks completed successfully.")


agent = NetworkPlanningAgent()
agent.execute("Deploy BGP route reflector cluster with 2 RR clients in AS 65001")


---
## Demo 2 — ReAct with Lookahead: Form a Hypothesis First

Standard ReAct loops are *reactive* — they try things and see what happens.

**ReAct + Lookahead** adds an upfront thinking phase:
1. **Hypothesize** — "What is most likely causing this?"
2. **Plan** — "What evidence would confirm or deny it?"
3. **Gather** — Execute the evidence-collection steps
4. **Conclude** — Confirm or revise the hypothesis

This is how **experienced network engineers** troubleshoot. They don't run
`show ip route` randomly — they form a mental model first, then test it.


In [ ]:
# ── ReAct with Lookahead ──────────────────────────────────────────────────────

# Simulated network state (what tools would return)
NETWORK_STATE = {
    "ospf_neighbors": "R1: FULL/DR, R2: EXSTART, R3: FULL/BDR",
    "interface_status": "Gi0/0: up/up, Gi0/1: up/up, Lo0: up/up",
    "mtu_check":  "Gi0/0 MTU=1500, R2-facing Gi0/1 MTU=1492 ← MISMATCH",
    "cpu_usage":  "5-min avg: 12%  — normal",
    "memory":     "Free: 256MB / Total: 512MB — normal",
    "ospf_timers": "Hello 10s Dead 40s — matches all neighbors",
    "auth_config": "No authentication configured",
    "log_tail":   "OSPF: Too many retransmissions on Gi0/1 to R2",
}

TOOLS = {
    "check_ospf_neighbors": lambda: NETWORK_STATE["ospf_neighbors"],
    "check_interface_status": lambda: NETWORK_STATE["interface_status"],
    "check_mtu": lambda: NETWORK_STATE["mtu_check"],
    "check_cpu": lambda: NETWORK_STATE["cpu_usage"],
    "check_memory": lambda: NETWORK_STATE["memory"],
    "check_ospf_timers": lambda: NETWORK_STATE["ospf_timers"],
    "check_auth": lambda: NETWORK_STATE["auth_config"],
    "check_logs": lambda: NETWORK_STATE["log_tail"],
}

def react_with_lookahead(problem: str):
    print(f"Problem: {problem}")
    print("=" * 60)

    # ── Phase 1: Lookahead — form hypothesis and investigation plan ───────────
    plan_prompt = f"""Network problem: {problem}

Available tools: {list(TOOLS.keys())}

Before gathering evidence, form a structured investigation plan:
1. What are the top 3 most likely root causes (ranked by probability)?
2. For each cause, list exactly which tools to run to confirm or deny it.
3. What is the fastest path to confirmation (minimum tools needed)?

Be concise. Format as:
HYPOTHESIS 1: <cause>
  Probability: high/medium/low
  Test with: <tool1>, <tool2>

HYPOTHESIS 2: ...

FASTEST PATH: <tool sequence>
"""
    print("\n[Phase 1] Forming hypothesis and plan...")
    plan = ask(plan_prompt, model=SONNET)
    print(plan)

    # ── Phase 2: Extract the fastest path tools ───────────────────────────────
    # Simple extraction: find tool names mentioned in the plan
    fast_tools = [t for t in TOOLS if t in plan]
    if not fast_tools:
        fast_tools = list(TOOLS.keys())[:3]  # fallback

    # ── Phase 3: Gather evidence ──────────────────────────────────────────────
    print(f"\n[Phase 2] Gathering evidence with: {fast_tools}")
    evidence = {}
    for tool_name in fast_tools:
        result = TOOLS[tool_name]()
        evidence[tool_name] = result
        print(f"  {tool_name}: {result}")

    # ── Phase 4: Conclude ─────────────────────────────────────────────────────
    conclude_prompt = f"""Problem: {problem}

Evidence gathered:
{json.dumps(evidence, indent=2)}

Based on this evidence:
1. What is the confirmed root cause?
2. What is the fix?
3. How confident are you (0-100%)?
"""
    print("\n[Phase 3] Concluding...")
    conclusion = ask(conclude_prompt, model=SONNET)
    print(conclusion)

react_with_lookahead("OSPF adjacency stuck in EXSTART with neighbor R2")


---
## Demo 3 — Plan-and-Execute: Separate Thinking from Acting

**Plan-and-Execute** uses *two separate agents*:

| Agent | Role | Model |
|-------|------|-------|
| **Planner** | Creates the full step-by-step plan upfront | Sonnet (smart) |
| **Reviewer** | Validates the plan for safety risks | Sonnet |
| **Executor** | Carries out each step, reports results | Haiku (fast) |

The Planner *never* executes. The Executor *never* plans. This separation means:
- You can review the plan before anything runs (like a **change freeze** gate)
- You can re-plan mid-execution if a step fails
- Cheaper execution (Haiku) with smarter planning (Sonnet)

This maps directly to how network change management works:
**RFCs (Request for Change) → Change Advisory Board → Maintenance window execution.**


In [ ]:
# ── Plan-and-Execute Architecture ─────────────────────────────────────────────

class PlanAndExecuteAgent:
    """
    Three-role separation:
    Planner (Sonnet) → Reviewer (Sonnet) → Executor (Haiku)
    """

    def __init__(self):
        self.plan     = []
        self.results  = []

    # ── Role 1: Planner ───────────────────────────────────────────────────────
    def create_plan(self, goal: str) -> list[dict]:
        print(f"[Planner] Creating plan for: {goal}")
        prompt = f"""You are a network change management planner.

Create a safe, step-by-step execution plan for:
{goal}

Return ONLY a JSON array where each step has:
  - "step": integer (1-based)
  - "action": what to do (one sentence)
  - "command": the CLI command or API call to run
  - "expected": what success looks like
  - "rollback": how to undo this step if it fails

Keep it to 5-7 steps.
"""
        raw  = ask(prompt, model=SONNET)
        match = re.search(r'\[.*\]', raw, re.DOTALL)
        if not match:
            raise ValueError("Planner returned no JSON")
        self.plan = json.loads(match.group())
        return self.plan

    # ── Role 2: Reviewer ──────────────────────────────────────────────────────
    def review_plan(self) -> dict:
        print("\n[Reviewer] Checking plan for risks...")
        plan_text = json.dumps(self.plan, indent=2)
        prompt = f"""You are a senior network engineer reviewing a change plan.

Plan to review:
{plan_text}

Identify:
1. Any steps that could cause outages if they fail
2. Missing safety checks
3. Steps that need a maintenance window
4. Overall risk level: LOW / MEDIUM / HIGH

Be concise. End with: APPROVED or NEEDS_REVISION
"""
        review = ask(prompt, model=SONNET)
        approved = "APPROVED" in review.upper()
        return {"review": review, "approved": approved}

    # ── Role 3: Executor ──────────────────────────────────────────────────────
    def execute_step(self, step: dict) -> str:
        prompt = f"""You are executing a network change step.

Step {step['step']}: {step['action']}
Command: {step['command']}
Expected result: {step['expected']}

Simulate executing this command and report:
- Output (realistic CLI-style output)
- Success/Failure
- Next action if failed

Keep response to 3-4 lines.
"""
        return ask(prompt, model=HAIKU)

    # ── Full workflow ─────────────────────────────────────────────────────────
    def run(self, goal: str):
        print("=" * 60)

        # Plan
        plan = self.create_plan(goal)
        print(f"\nPlan created — {len(plan)} steps:")
        for s in plan:
            print(f"  Step {s['step']}: {s['action']}")

        # Review
        review_result = self.review_plan()
        print(f"\n[Review Result]")
        print(review_result["review"])

        if not review_result["approved"]:
            print("\n⛔ Plan NOT approved. Stopping before execution.")
            return

        # Execute
        print("\n[Executor] Starting execution...")
        for step in plan:
            print(f"\n  ── Step {step['step']}: {step['action']}")
            result = self.execute_step(step)
            self.results.append({"step": step["step"], "result": result})
            print(f"  {result}")

        print(f"\n✅ Execution complete. {len(self.results)} steps ran.")


agent = PlanAndExecuteAgent()
agent.run("Migrate OSPF area 0 backbone to use SHA-256 authentication")


---
## Demo 4 — Reflexion: Agents That Learn from Failure

**Reflexion** is verbal reinforcement learning — the agent improves through
*language*, not gradient descent.

After each failure, instead of retraining the model, the agent:
1. **Reflects** — "Why did I fail? What assumption was wrong?"
2. **Stores the reflection** as memory
3. **Re-plans** using the learned lesson

This is like a network engineer keeping a **post-mortem document** and checking
it before every similar change. The lesson from the last outage shapes the
approach to the next one.

```
Trial 1: [Agent attempts fix] → Fails
Reflection: "I assumed the problem was in L3. Forgot to check L2 first."

Trial 2: [Agent checks L2 first] → Fails
Reflection: "MTU was fine. Need to check OSPF DR election."

Trial 3: [Agent checks DR] → Success! ✓
```


In [ ]:
# ── Reflexion: Verbal Reinforcement Learning ───────────────────────────────────

class ReflexionAgent:
    """
    Actor-Evaluator-Reflector pattern.
    Memories (reflections) persist across trials.
    """

    def __init__(self, max_trials=3):
        self.max_trials = max_trials
        self.memory     = []   # grows with each failed trial

    # ── Actor: attempt to solve the problem ──────────────────────────────────
    def act(self, problem: str, trial: int) -> str:
        memory_block = ""
        if self.memory:
            memory_block = "\n\nYour past attempts and reflections:\n"
            for i, m in enumerate(self.memory, 1):
                memory_block += f"Trial {i}: {m}\n"

        prompt = f"""You are a network troubleshooting agent.

Problem: {problem}
Trial: {trial} of {self.max_trials}
{memory_block}

Propose a specific diagnostic action and what you expect to find.
If you have memory from past trials, use those lessons.
Format:
ACTION: <what to do>
EXPECTED: <what you hope to find>
REASONING: <why this makes sense given past failures>
"""
        return ask(prompt, model=SONNET)

    # ── Evaluator: judge success/failure ──────────────────────────────────────
    def evaluate(self, action: str, trial: int) -> dict:
        # Simulate: first 2 trials fail, third succeeds
        # (In real use, this would run actual network commands)
        success = (trial >= 3)
        feedback = {
            1: "Action taken. No improvement. OSPF still stuck in EXSTART. MTU appears fine on surface check.",
            2: "Action taken. Slight progress — adjacency flapped but returned to EXSTART. Issue persists.",
            3: "Action taken. Adjacency moved to FULL. Root cause: asymmetric MTU on the P2P link — one side was set in software, other needed hardware-level config.",
        }.get(trial, "Unknown outcome.")
        return {"success": success, "feedback": feedback}

    # ── Reflector: generate verbal reflection ────────────────────────────────
    def reflect(self, problem: str, action: str, feedback: str) -> str:
        prompt = f"""You are analyzing a failed network troubleshooting attempt.

Problem: {problem}
What you tried: {action}
What happened: {feedback}

Write a brief reflection (2-3 sentences) covering:
1. What assumption was wrong?
2. What should you try next time?
3. What will you check earlier?
"""
        return ask(prompt, model=HAIKU)

    # ── Full loop ─────────────────────────────────────────────────────────────
    def run(self, problem: str):
        print(f"Problem: {problem}")
        print("=" * 60)

        for trial in range(1, self.max_trials + 1):
            print(f"\n[Trial {trial}/{self.max_trials}]")

            # Act
            action = self.act(problem, trial)
            print(f"\nAction proposed:\n{action}")

            # Evaluate
            result = self.evaluate(action, trial)
            print(f"\nOutcome: {'✓ SUCCESS' if result['success'] else '✗ FAILED'}")
            print(f"Feedback: {result['feedback']}")

            if result["success"]:
                print(f"\n✅ Problem solved in {trial} trial(s)!")
                print(f"\nMemory bank ({len(self.memory)} lessons):")
                for i, m in enumerate(self.memory, 1):
                    print(f"  Lesson {i}: {m}")
                return

            # Reflect
            print("\n[Reflecting on failure...]")
            reflection = self.reflect(problem, action, result["feedback"])
            self.memory.append(f"FAILED: {reflection}")
            print(f"Reflection stored: {reflection}")

        print(f"\n⚠ Max trials ({self.max_trials}) reached without resolution.")


agent = ReflexionAgent(max_trials=3)
agent.run("OSPF neighbor stuck in EXSTART after hardware replacement")


---
## Demo 5 — Full Pipeline: All Architectures for P1 Incident Response

In production, you don't pick *one* architecture — you **compose** them:

```
P1 Incident Alert
       │
   [HTN Planner] → Decompose into: investigate → mitigate → verify → report
       │
   [ReAct + Lookahead] → For "investigate": form hypothesis, gather evidence
       │
   [Reflexion Memory] → Apply lessons from past similar incidents
       │
   [Plan-and-Execute] → For "mitigate": plan → review → execute
       │
   Incident Closed ✓
```

This demo orchestrates all four patterns for a realistic P1 outage scenario.


In [ ]:
# ── Full Production Pipeline ──────────────────────────────────────────────────

def run_p1_pipeline(incident: str):
    print(f"P1 INCIDENT: {incident}")
    print("=" * 70)
    start = time.time()

    # ── Stage 1: HTN — decompose incident response into phases ────────────────
    print("\n[Stage 1] HTN Decomposition — Breaking down incident response phases")
    decompose_prompt = f"""Network P1 incident: {incident}

Decompose into 4 high-level phases with dependencies:
[
  {{"name": "investigate", "description": "...", "depends_on": []}},
  {{"name": "mitigate",    "description": "...", "depends_on": ["investigate"]}},
  {{"name": "verify",      "description": "...", "depends_on": ["mitigate"]}},
  {{"name": "report",      "description": "...", "depends_on": ["verify"]}}
]
Return ONLY the JSON array.
"""
    raw = ask(decompose_prompt, model=SONNET)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    phases = json.loads(match.group()) if match else [
        {"name": "investigate", "description": "Gather symptoms and identify root cause.", "depends_on": []},
        {"name": "mitigate",    "description": "Apply temporary fix to restore service.", "depends_on": ["investigate"]},
        {"name": "verify",      "description": "Confirm service is restored.", "depends_on": ["mitigate"]},
        {"name": "report",      "description": "Document findings and timeline.", "depends_on": ["verify"]},
    ]
    for p in phases:
        print(f"  Phase: {p['name']} — {p['description']}")

    # ── Stage 2: ReAct+Lookahead — investigation phase ────────────────────────
    print("\n[Stage 2] ReAct+Lookahead — Investigating root cause")
    investigate_prompt = f"""P1 incident: {incident}

Form a hypothesis and the top 3 evidence checks needed.
Format:
HYPOTHESIS: <most likely cause>
CHECK 1: <what to look at>
CHECK 2: <what to look at>
CHECK 3: <what to look at>
CONFIDENCE: <0-100%>
"""
    hypothesis = ask(investigate_prompt, model=SONNET)
    print(hypothesis)

    # ── Stage 3: Reflexion Memory — apply past lessons ────────────────────────
    print("\n[Stage 3] Reflexion — Applying lessons from past incidents")
    memory_prompt = f"""Based on this P1 incident: {incident}

And this hypothesis: {hypothesis}

Generate 2 "lessons from past similar incidents" that an experienced
network engineer would remember. Format as:
LESSON 1: <specific actionable lesson>
LESSON 2: <specific actionable lesson>
"""
    lessons = ask(memory_prompt, model=HAIKU)
    print(lessons)

    # ── Stage 4: Plan-and-Execute — mitigation phase ──────────────────────────
    print("\n[Stage 4] Plan-and-Execute — Mitigation plan")
    mitigate_prompt = f"""Create a 3-step mitigation plan for: {incident}

Hypothesis confirmed: {hypothesis.split(chr(10))[0]}
Lessons applied: {lessons}

Return JSON:
[
  {{"step": 1, "action": "...", "command": "...", "rollback": "..."}},
  {{"step": 2, "action": "...", "command": "...", "rollback": "..."}},
  {{"step": 3, "action": "...", "command": "...", "rollback": "..."}}
]
"""
    raw2  = ask(mitigate_prompt, model=SONNET)
    match2 = re.search(r'\[.*\]', raw2, re.DOTALL)
    mplan  = json.loads(match2.group()) if match2 else []
    print("Mitigation steps:")
    for s in mplan:
        print(f"  Step {s.get('step','?')}: {s.get('action','')}")
        print(f"    Rollback: {s.get('rollback','')}")

    # ── Stage 5: Verify and close ─────────────────────────────────────────────
    print("\n[Stage 5] Verify and close incident")
    verify_prompt = f"""Incident: {incident}
Mitigation applied: {[s.get('action') for s in mplan]}

Write a 2-sentence verification checklist and incident close statement.
"""
    closure = ask(verify_prompt, model=HAIKU)
    print(closure)

    elapsed = time.time() - start
    print(f"\n{'='*70}")
    print(f"P1 incident pipeline complete in {elapsed:.1f}s")
    print(f"Phases executed: {' → '.join(p['name'] for p in phases)}")
    print(f"Architecture used: HTN + ReAct+Lookahead + Reflexion + Plan-and-Execute")


run_p1_pipeline("BGP session drops affecting 40% of customer prefixes on core router R-CORE-01")


---
## Summary — Choosing the Right Architecture

| You need to… | Use |
|---|---|
| Break a complex goal into ordered steps | **HTN + DAG** |
| Troubleshoot without running random commands | **ReAct + Lookahead** |
| Separate plan review from execution (high-risk changes) | **Plan-and-Execute** |
| Improve without retraining the model | **Reflexion** |
| Handle a real P1 with all complexity | **Compose all four** |

### Key Takeaways

1. **Planning > Reacting** — Form a hypothesis before grabbing tools. Senior engineers do this naturally.
2. **Separate roles** — Planner, Reviewer, Executor. Cheap model (Haiku) for execution, smart model (Sonnet) for planning.
3. **Verbal memory** — Reflexion stores lessons in language. No GPU, no retraining — just prompts.
4. **DAG guarantees order** — Topological sort ensures you never skip prerequisite steps.

### What's Next

- **Chapter 20**: Multi-Agent Systems — orchestrating fleets of specialized agents that collaborate like a NOC team
- **Chapter 21**: Production Safety — guardrails, circuit breakers, and human-in-the-loop gates for autonomous agents

> The journey from "chatbot" to "production-grade network automation" runs through these architectures. Master the patterns, not just the API calls.
